# Behavioral cloning: optional Colab training

This notebook reuses the project's Python modules. It does not contain a separate CNN or preprocessing implementation. No Colab execution is claimed by this unexecuted notebook.

Choose a GPU runtime if available; CPU is supported. Prepare a ZIP containing the project source, config.py, requirements files and data/raw (the real labelled images). Exclude .venv, .test-tmp folders, old models/reports and machine-specific processed manifests. Keep provenance.json with the dataset. For the full dataset, upload/download it separately and select one verified driving log. Do not combine duplicated tracks.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile
import zipfile
from google.colab import files

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one project ZIP.')
workspace = Path(tempfile.mkdtemp(prefix='behavioral-cloning-', dir='/content'))
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        destination = (workspace / member.filename).resolve()
        if not destination.is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe archive path.')
    archive.extractall(workspace)
roots = [p.parent for p in workspace.rglob('config.py') if (p.parent / 'src/train.py').is_file()]
if len(roots) != 1:
    raise ValueError('Expected one project folder containing config.py and src/train.py.')
project = roots[0]
os.chdir(project)
print('Project:', project)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-r', 'requirements-model.txt'], check=True)
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Available devices:', tf.config.list_physical_devices())
print('GPU will be used automatically when supported; otherwise CPU.')

## Select the real dataset and prepare local paths

For several logs, edit CSV_PATH to select one explicitly. No labels are synthesized. Use verified session IDs when available. The included 300-frame excerpt is a limited demonstration dataset, not evidence of robust driving. Keep this split fixed for all validation comparisons.

In [ ]:
from src.data_loader import find_driving_log
from src.explore_data import analyze_dataset
from src.prepare_data import prepare_dataset

CSV_PATH = None  # Example: project / 'data/raw/lab_subset/driving_log.csv'
selected_csv = find_driving_log(project / 'data/raw', CSV_PATH)
processed = project / 'data/colab_processed'
models = project / 'models/colab_run'
reports = project / 'reports/colab_run'
analysis = analyze_dataset(selected_csv, reports_dir=reports, processed_dir=processed)
split_summary = prepare_dataset(selected_csv, output_dir=processed)
print(json.dumps({'validation': analysis['valid_records'], 'splits': split_summary}, indent=2))

## Train the shared milestone CNN

Defaults match local training: Adam 0.001, MSE, batch 32, maximum 20 epochs, patience 4. Training alone uses augmentation. A rerun protects existing models: choose a new models/reports directory for another experiment. Use validation for selection; do not tune with the test set.

In [ ]:
from src.train import train_model
metadata = train_model(processed_dir=processed, models_dir=models, reports_dir=reports,
                       epochs=20, batch_size=32, learning_rate=0.001, patience=4, seed=42)
print('Best epoch:', metadata['best_epoch'])
print('Best validation MSE:', metadata['best_validation_loss'])

## Final evaluation and reload verification

Run after choosing the model using validation. Baseline and CNN share the same held-out records. Results may be poor; report them faithfully.

In [ ]:
from src.evaluate import evaluate_model
from src.predict import load_trained_model, predict_steering
import pandas as pd

metrics = evaluate_model(processed, models / 'best_model.keras', reports)
print(json.dumps(metrics, indent=2))
model, saved_metadata = load_trained_model(models / 'best_model.keras')
test_records = pd.read_csv(processed / 'test.csv')
print('Reloaded model prediction:', predict_steering(test_records.iloc[0].image_path, model))

In [ ]:
from IPython.display import display, Image
for name in ['training_loss.png', 'actual_vs_predicted.png', 'error_distribution.png', 'sample_predictions.png']:
    display(Image(filename=str(reports / 'figures' / name)))

## Export to Windows

Download the ZIP and extract its models/ and reports/ into a separate copy of the local project first, preserving older experiment artifacts. Both best_model.keras and model_metadata.json are required. Compare runtime versions in metadata with local TensorFlow/Keras; use compatible versions if needed. The same saved preprocessing is automatically loaded by Streamlit.

Colab prediction CSV image paths are local to Colab. For local reevaluation/sample inspection, prepare the same raw dataset with the same split settings into a new local processed directory and run src.evaluate with --processed-dir pointing there. Content/label signatures must match. Merely reloading the identical checkpoint for transport verification is not hyperparameter selection. Never silently reuse stale absolute paths.

Launch locally: python -m streamlit run app.py. Colab training does not establish independent test drives or safe driving.

In [ ]:
versions = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(reports / 'colab_requirements.txt').write_text(versions, encoding='utf-8')
export_path = workspace / 'trained_artifacts.zip'
with zipfile.ZipFile(export_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in models.iterdir():
        if path.is_file():
            archive.write(path, Path('models') / path.name)
    for path in reports.rglob('*'):
        if path.is_file():
            archive.write(path, Path('reports') / path.relative_to(reports))
files.download(str(export_path))